# Dataset 2 – City Activities and Planned Works

This notebook documents the acquisition, data understanding and initial data-quality review for Dataset 2 in the AgeTogether FIT5120 Iteration 1 project. It deliberately does **not** apply product-relevance filtering, create a processed export, or make recommendations.

## 1. Data Acquisition and Provenance

**Dataset:** City Activities and Planned Works  
**Provider:** City of Melbourne Open Data, published through DataVic  
**Official dataset page:** https://discover.data.vic.gov.au/dataset/city-activities-and-planned-works  
**Official resource page:** https://discover.data.vic.gov.au/en_AU/dataset/city-activities-and-planned-works/resource/f3e6a6af-a0b7-4c0e-a7f8-1de0fdb183e2  
**Resource download URL used:** https://discover.data.vic.gov.au/datastore/dump/f3e6a6af-a0b7-4c0e-a7f8-1de0fdb183e2  
**Licence:** CC BY 4.0  
**Local acquisition date:** 3 September 2026  
**Official data last updated:** 8 February 2024 (according to the DataVic resource metadata)  
**Official metadata last updated:** 13 June 2024  
**Update frequency:** Not available from the source / listed as Unknown in the dataset metadata  
**Raw-file SHA-256:** `40f5febb3a39356119c1ae3d012185913f8847f1d06f4081a671bb1a82e93c88`

This official City of Melbourne open-data dataset was selected because its `classification`, expected date range, location and spatial fields provide evidence that some permit records are event-related. However, it is a **permit-based activity dataset**, not a public event catalogue: it also contains city works, traffic management, reserved parking and structures records.

AgeTogether must therefore not automatically treat every record as a public social activity. The later product-relevance decision must be transparent and must preserve this limitation. The raw file is retained unchanged under `data/raw/` for reproducibility.

## 2. Data Understanding

The raw CSV is loaded from a repository-relative path. The helper below locates the repository root without depending on a user-specific absolute path.

In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd


def display(value):
    print(value.to_string())


RAW_FILENAME = "city_activities_and_planned_works.csv"
ACQUISITION_DATE = pd.Timestamp("2026-09-03")


def find_repository_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "data" / "raw" / RAW_FILENAME).exists():
            return candidate
    raise FileNotFoundError("Repository root containing the Dataset 2 raw CSV was not found.")


repo_root = find_repository_root(Path.cwd().resolve())
raw_path = repo_root / "data" / "raw" / RAW_FILENAME
df = pd.read_csv(raw_path)

with raw_path.open("rb") as raw_file:
    raw_sha256 = hashlib.sha256(raw_file.read()).hexdigest()

print(f"Repository root: {repo_root}")
print(f"Raw file: {raw_path.relative_to(repo_root)}")
print(f"SHA-256: {raw_sha256}")

### 2.1 Dataset Overview

In [ ]:
print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types:")
display(df.dtypes.rename("data_type").to_frame())
print("\nFirst five raw records:")
display(df.head())

### 2.2 Categorical Field Inventory

`classification` and `status` are the primary product-relevant categorical fields. `small_area` and `source_id` are also inspected because they describe locality and permit/source references. A high number of `source_id` values is expected for a reference field, so it is summarised rather than used as a category filter.

In [ ]:
for column in ["classification", "status", "small_area"]:
    print(f"{column}: {df[column].nunique(dropna=True)} unique non-missing values")
    display(
        df[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="record_count")
    )

print(f"source_id: {df['source_id'].nunique(dropna=True)} unique non-missing values")

## 3. Initial Data Quality Checks

These checks describe the raw data; they do not remove or relabel records. Missing values mean the source does not provide a value for that field, not that the corresponding real-world attribute is negative or absent.

### 3.1 Missing Values and Duplicates

In [ ]:
missing_summary = (
    df.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_percentage=lambda table: table["missing_count"] / len(df) * 100)
)
display(missing_summary)

print(f"Exact duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate activity_id values: {df['activity_id'].duplicated().sum()}")

### 3.2 Date Parsing and Expected Activity Ranges

The source fields are parsed into temporary datetime series for validation only. The source documentation states that `start_date` and `end_date` are **expected permit/activity ranges**, not guaranteed actual event durations. A range longer than 365 days is flagged only as an unusually long range for review; it is not deleted or treated as an error automatically.

In [ ]:
start_parsed = pd.to_datetime(df["start_date"], errors="coerce")
end_parsed = pd.to_datetime(df["end_date"], errors="coerce")
date_range_days = (end_parsed - start_parsed).dt.days

date_quality_summary = pd.DataFrame(
    {
        "metric": [
            "start_date parsed successfully",
            "start_date parsing failures",
            "end_date parsed successfully",
            "end_date parsing failures",
            "end_date earlier than start_date",
            "ranges longer than 365 days",
            "minimum parsed date range (days)",
            "maximum parsed date range (days)",
            "median parsed date range (days)",
        ],
        "value": [
            start_parsed.notna().sum(),
            start_parsed.isna().sum(),
            end_parsed.notna().sum(),
            end_parsed.isna().sum(),
            (end_parsed < start_parsed).sum(),
            (date_range_days > 365).sum(),
            date_range_days.min(),
            date_range_days.max(),
            date_range_days.median(),
        ],
    }
)
display(date_quality_summary)

temporal_status = np.select(
    [
        end_parsed < ACQUISITION_DATE,
        start_parsed > ACQUISITION_DATE,
        (start_parsed <= ACQUISITION_DATE) & (end_parsed >= ACQUISITION_DATE),
    ],
    ["expired", "future", "current expected range"],
    default="unclassified because of date parsing failure",
)

temporal_summary = (
    pd.Series(temporal_status, name="temporal_status")
    .value_counts()
    .rename_axis("temporal_status")
    .reset_index(name="record_count")
)
temporal_summary["percentage"] = temporal_summary["record_count"] / len(df) * 100
display(temporal_summary)

### 3.3 Location and Coordinate Validation

`geo_point_2d` stores latitude and longitude together as text. The values are split into temporary numeric series for validation only; the raw text field is preserved. Geographic validity is checked first against world coordinate limits, then against a broad Victoria bounding range (latitude −39.3 to −33.8, longitude 140.8 to 150.1). This is a validation range, not a product geographic filter.

In [ ]:
coordinate_parts = df["geo_point_2d"].str.split(",", n=1, expand=True)
latitude = pd.to_numeric(coordinate_parts[0].str.strip(), errors="coerce")
longitude = pd.to_numeric(coordinate_parts[1].str.strip(), errors="coerce")

coordinates_parsed = latitude.notna() & longitude.notna()
globally_valid_coordinates = latitude.between(-90, 90) & longitude.between(-180, 180)
victoria_range_coordinates = latitude.between(-39.3, -33.8) & longitude.between(140.8, 150.1)

coordinate_quality_summary = pd.DataFrame(
    {
        "metric": [
            "missing geo_point_2d",
            "coordinates parsed successfully",
            "coordinate parsing failures",
            "minimum latitude",
            "maximum latitude",
            "minimum longitude",
            "maximum longitude",
            "globally valid coordinate pairs",
            "coordinate pairs within broad Victoria validation range",
            "missing location",
        ],
        "value": [
            df["geo_point_2d"].isna().sum(),
            coordinates_parsed.sum(),
            (~coordinates_parsed).sum(),
            latitude.min(),
            latitude.max(),
            longitude.min(),
            longitude.max(),
            globally_valid_coordinates.sum(),
            victoria_range_coordinates.sum(),
            df["location"].isna().sum(),
        ],
    }
)
display(coordinate_quality_summary)

### 3.4 Classification and Status Inventory

In [ ]:
classification_summary = (
    df["classification"]
    .value_counts(dropna=False)
    .rename_axis("classification")
    .reset_index(name="record_count")
)
classification_summary["percentage"] = classification_summary["record_count"] / len(df) * 100
display(classification_summary)

status_summary = (
    df["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="record_count")
)
status_summary["percentage"] = status_summary["record_count"] / len(df) * 100
display(status_summary)

event_related_classifications = {"Event", "Public Event", "Private Event"}
event_related_count = df["classification"].isin(event_related_classifications).sum()
operational_count = df["classification"].isin(
    {"Structures", "Traffic Management", "Reserved Parking"}
).sum()

print(f"Event-related classification records: {event_related_count} ({event_related_count / len(df) * 100:.1f}%)")
print(f"Operational/administrative classification records: {operational_count} ({operational_count / len(df) * 100:.1f}%)")

## 4. Initial Observations

The raw dataset contains 605 records and six classification values. `Event`, `Public Event` and `Private Event` are present, together accounting for **36 records (6.0%)**. This is evidence that the source contains event-related permits, but it does not establish that any record is a public, current or suitable AgeTogether social activity.

The remaining **569 records (94.0%)** have `Structures`, `Traffic Management` or `Reserved Parking` classifications. These categories indicate a large operational/administrative component and justify a later transparent relevance-review stage. No filtering is applied in this notebook yet.

There is no dedicated public event-title field in the actual schema. `classification` is a broad label, `location` is a site/area field, and `notes` is missing for 105 records, including all 36 records with an event-related classification in this acquired file. `activity_id` and `source_id` appear to be technical permit/reference identifiers and must not be presented as public event titles.

All 605 `geo_point_2d` values parse successfully and fall within global and broad Victoria validation ranges. However, the source documentation describes the geometry as a general activity area; it must not be presented as a precise venue entrance.

Date quality requires caution. All `start_date` values parse, but one `end_date` fails parsing. Forty-three expected ranges exceed 365 days, with a maximum parsed range of 33,237 days. Relative to the local acquisition date (3 September 2026), 599 records have an expired expected end date, five records have a current expected range, and one cannot be classified because its end date does not parse. The current expected ranges include anomalously long permit ranges. This confirms a major freshness limitation for any future product use.

At this stage, no record is labelled suitable for older adults or publicly accessible. The source does not provide verified public access, event title, public event URL, price, organiser, booking, accessibility or actual live availability fields. Missing values must not be interpreted as negative values, and none of these attributes will be invented.

## 5. Event-Related Record Investigation

The event-related subset is defined only by the source `classification` values `Event`, `Public Event` and `Private Event`. This is an investigation view, not a public-event claim.

The actual schema has no dedicated event-title field. `activity_id` and `source_id` are permit/reference identifiers, `classification` is a generic type label, and `location` is a source-provided area or site description. For the 36 event-related records, `location` is complete and contains 33 unique human-readable values. It is therefore suitable as a **location label** in an Iteration 1 prototype, but it must not be renamed or presented as an event name.

The transparent display approach for a future frontend is:

- generic heading: `classification` (for example, “Public Event”);
- main location text: `location`;
- supporting details: expected `start_date`, `end_date`, `small_area` and `status`, clearly labelled as permit/activity data;
- source attribution: City of Melbourne – City Activities and Planned Works.

This approach does not invent an event title or imply public availability, booking, accessibility, price, organiser or live status.

In [ ]:
event_related_classifications = {"Event", "Public Event", "Private Event"}
event_related_df = df.loc[
    df["classification"].isin(event_related_classifications),
    [
        "activity_id",
        "classification",
        "location",
        "small_area",
        "start_date",
        "end_date",
        "status",
        "source_id",
    ],
].copy()

print(f"Event-related records: {len(event_related_df)}")
print(f"Non-missing location values: {event_related_df['location'].notna().sum()}")
print(f"Unique location values: {event_related_df['location'].nunique(dropna=True)}")
print(f"Non-missing notes values: {df.loc[event_related_df.index, 'notes'].notna().sum()}")

print("\nClassification distribution:")
display(
    event_related_df["classification"]
    .value_counts()
    .rename_axis("classification")
    .reset_index(name="record_count")
)

print("\nStatus distribution:")
display(
    event_related_df["status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="record_count")
)

print("\nAll event-related records:")
display(event_related_df.sort_values(["classification", "start_date", "location"]))

## 6. Iteration 1 Relevance Filtering

This step creates a transparent product-relevance layer while retaining all 605 source records. The rule is intentionally simple and depends only on the actual source `classification` field.

| Source classification | prototype_relevance | Rule explanation |
|---|---|---|
| `Public Event` | Included | Included because the source classification is `Public Event`. |
| `Event` | Included | Included because the source classification is `Event`. |
| `Private Event` | Excluded | Excluded because the source classification is `Private Event`. |
| `Structures` | Excluded | Excluded because the source classification is `Structures`. |
| `Traffic Management` | Excluded | Excluded because the source classification is `Traffic Management`. |
| `Reserved Parking` | Excluded | Excluded because the source classification is `Reserved Parking`. |

Freshness is **not** an Iteration 1 exclusion rule. Expected date ranges are preserved as source evidence because this iteration prioritises relevant open data, reproducible processing and traceable provenance. The known freshness limitation remains documented.

`Included` means only that a record is sufficiently event-related to demonstrate the Iteration 1 Activities data pipeline. It does **not** mean currently available, upcoming, suitable for older adults, publicly accessible (except that `Public Event` is the source classification), bookable, recommended, safe or accessible.

In [ ]:
included_classifications = {"Event", "Public Event"}

activity_classified_df = df.copy()
coordinate_parts = activity_classified_df["geo_point_2d"].str.split(",", n=1, expand=True)
activity_classified_df["latitude"] = pd.to_numeric(coordinate_parts[0].str.strip(), errors="coerce")
activity_classified_df["longitude"] = pd.to_numeric(coordinate_parts[1].str.strip(), errors="coerce")
activity_classified_df["prototype_relevance"] = np.where(
    activity_classified_df["classification"].isin(included_classifications),
    "Included",
    "Excluded",
)
activity_classified_df["prototype_relevance_reason"] = activity_classified_df["classification"].map(
    lambda value: f"{'Included' if value in included_classifications else 'Excluded'} because source classification is {value}."
)
activity_classified_df["source_dataset"] = "City of Melbourne – City Activities and Planned Works"

prototype_summary = (
    activity_classified_df["prototype_relevance"]
    .value_counts()
    .rename_axis("prototype_relevance")
    .reset_index(name="record_count")
)
prototype_summary["percentage"] = prototype_summary["record_count"] / len(activity_classified_df) * 100
display(prototype_summary)

included_df = activity_classified_df.loc[
    activity_classified_df["prototype_relevance"].eq("Included")
].copy()

print("\nIncluded classification distribution:")
display(
    included_df["classification"]
    .value_counts()
    .rename_axis("classification")
    .reset_index(name="record_count")
)

included_completeness = pd.DataFrame(
    {
        "field_or_check": [
            "location non-missing",
            "latitude non-missing",
            "longitude non-missing",
            "start_date non-missing",
            "end_date non-missing",
        ],
        "complete_record_count": [
            included_df["location"].notna().sum(),
            included_df["latitude"].notna().sum(),
            included_df["longitude"].notna().sum(),
            included_df["start_date"].notna().sum(),
            included_df["end_date"].notna().sum(),
        ],
    }
)
included_completeness["percentage"] = included_completeness["complete_record_count"] / len(included_df) * 100
print("\nIncluded record completeness:")
display(included_completeness)

## 7. Product Data View and Export

Two non-destructive outputs are created:

1. `agetogether_activity_records_classified.csv` contains all source records plus the transparent prototype-relevance fields, parsed latitude/longitude and source attribution. It is retained as DS audit evidence.
2. `agetogether_activity_prototype.csv` contains only records classified as `Included`: `Event` and `Public Event`.

The prototype export intentionally contains no fabricated event title, booking URL, price, organiser, accessibility or description. It remains permit-based prototype activity data, not a confirmation of a live public-event catalogue.

In [ ]:
classified_export_path = repo_root / "data" / "processed" / "agetogether_activity_records_classified.csv"
prototype_export_path = repo_root / "data" / "processed" / "agetogether_activity_prototype.csv"

prototype_export_columns = [
    "activity_id",
    "classification",
    "location",
    "small_area",
    "start_date",
    "end_date",
    "status",
    "latitude",
    "longitude",
    "source_id",
    "source_dataset",
]

activity_classified_df.to_csv(classified_export_path, index=False)
activity_prototype_df = included_df[prototype_export_columns].copy()
activity_prototype_df.to_csv(prototype_export_path, index=False)

print(f"Classified export: {classified_export_path.relative_to(repo_root)} ({len(activity_classified_df)} records)")
print(f"Prototype export: {prototype_export_path.relative_to(repo_root)} ({len(activity_prototype_df)} records)")
print("\nPrototype export columns:")
print(activity_prototype_df.columns.tolist())
display(activity_prototype_df.head())

### 7.1 Export Validation

The following checks confirm that the raw source remains unchanged, all source records remain available in the audit export, and the prototype export contains only the documented `Event` and `Public Event` classifications.

In [ ]:
classified_export_df = pd.read_csv(classified_export_path)
prototype_export_df = pd.read_csv(prototype_export_path)

assert raw_sha256 == "40f5febb3a39356119c1ae3d012185913f8847f1d06f4081a671bb1a82e93c88"
assert len(df) == 605
assert len(activity_classified_df) == 605
assert len(classified_export_df) == 605
assert len(activity_prototype_df) == 35
assert len(prototype_export_df) == 35
assert set(prototype_export_df["classification"].unique()) == {"Event", "Public Event"}
assert activity_classified_df["prototype_relevance"].value_counts().to_dict() == {"Excluded": 570, "Included": 35}
assert activity_classified_df["activity_id"].equals(df["activity_id"])
assert activity_classified_df["latitude"].notna().all()
assert activity_classified_df["longitude"].notna().all()

print("Validation passed:")
print("- Raw SHA-256 matches the acquisition checksum.")
print("- Raw and classified datasets each contain 605 records.")
print("- Prototype dataset contains 35 Event/Public Event records.")
print("- Prototype export has only documented source-supported fields.")
print("- No machine-specific absolute paths are used for input or output data paths.")